# High Performance Python: поза межами чистого Python

У цьому ноутбуці дивимось, як прискорювати Python-код, коли звичайних циклів уже не вистачає. Основні ідеї лекції: векторизація, обробка великих даних частинами, JIT/AOT-компіляція та zero-copy робота з памʼяттю.


## Чому чистий Python може бути повільним

Python зручний, але кожна ітерація циклу має накладні витрати інтерпретатора. Якщо даних багато, краще передати роботу бібліотеці, яка виконує критичну частину в оптимізованому C/LLVM/XLA-коді.


## Векторизація: мислення масивами

У першому прикладі порівнюємо звичайний Python-цикл і NumPy-операцію над цілим масивом. NumPy зберігає числа щільним блоком памʼяті й виконує операцію без Python-циклу по кожному елементу.


In [ ]:
import os
import time

import numpy as np
import psutil


def memory_usage_mb() -> float:
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024


size = 1_000_000
python_values = list(range(size))

start_mem = memory_usage_mb()
start = time.perf_counter()

python_result = []
for value in python_values:
    python_result.append(value * 2 + 5)

python_time = time.perf_counter() - start
python_mem = memory_usage_mb() - start_mem

numpy_values = np.arange(size)

start_mem = memory_usage_mb()
start = time.perf_counter()

numpy_result = numpy_values * 2 + 5

numpy_time = time.perf_counter() - start
numpy_mem = memory_usage_mb() - start_mem

print(f"Python loop time: {python_time:.4f} s")
print(f"Python memory   : {python_mem:.2f} MB")
print(f"NumPy time      : {numpy_time:.4f} s")
print(f"NumPy memory    : {numpy_mem:.2f} MB")
print("Same first 5:", python_result[:5], numpy_result[:5].tolist())


На що звернути увагу: Python-код явно проходить по кожному елементу, а NumPy описує операцію одразу для всього масиву. Це не просто коротший запис, а інший спосіб виконання.


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(42)
values = rng.integers(0, 10_000, size=2_000_000)

start = time.perf_counter()
python_filtered = [value for value in values if value % 3 == 0 and value > 5000 and value < 9000]
python_time = time.perf_counter() - start

start = time.perf_counter()
numpy_filtered = values[(values % 3 == 0) & (values > 5000) & (values < 9000)]
numpy_time = time.perf_counter() - start

print(f"Python filter time: {python_time:.4f} s")
print(f"NumPy filter time : {numpy_time:.4f} s")
print("Python count:", len(python_filtered))
print("NumPy count :", len(numpy_filtered))
print("First 5 NumPy:", numpy_filtered[:5])


Тут NumPy будує булеву маску і застосовує її до масиву. Такий підхід особливо корисний для фільтрації, агрегацій і математичних обчислень на великих наборах даних.


In [ ]:
import numpy as np

arr64 = np.arange(1_000_000, dtype=np.int64)
arr32 = np.arange(1_000_000, dtype=np.int32)
arr16 = np.arange(1_000_000, dtype=np.int16)

print("int64:", arr64.nbytes, "B")
print("int32:", arr32.nbytes, "B")
print("int16:", arr16.nbytes, "B")


Тип даних напряму впливає на памʼять. Якщо діапазон значень дозволяє, менший `dtype` зменшує обсяг даних і часто покращує кеш-локальність.


## Dask: коли дані більші за RAM

Dask схожий на Pandas за API, але не виконує обчислення одразу. Він будує граф задач і запускає його тільки після `.compute()`.


In [ ]:
import pandas as pd
import dask.dataframe as dd


pdf = pd.DataFrame({
    "country": ["US", "US", "DE", "FR"],
    "sales": [100, 200, 150, 300],
})

print("Pandas result:")
print(pdf.groupby("country")["sales"].mean())

if dd is None:
    print("\nDask не встановлено. Пропускаємо Dask-приклад.")
else:
    ddf = dd.from_pandas(pdf, npartitions=2)

    print("\nDask object before compute():")
    result = ddf.groupby("country")["sales"].mean()
    print(result)

    print("\nDask result after compute():")
    print(result.compute())


Що тут відбувається: Pandas одразу рахує результат, а Dask спершу повертає відкладений обʼєкт. Це важливо для великих файлів, які треба обробляти частинами.


In [ ]:
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import dask.dataframe as dd


num_rows = 100_000_000
countries = np.array(["USA", "Canada", "Germany", "France", "Ukraine", "Japan"])
rng = np.random.default_rng(7)

with tempfile.TemporaryDirectory() as temp_dir:
    csv_path = Path(temp_dir) / "sales.csv"

    df_source = pd.DataFrame({
        "country": rng.choice(countries, size=num_rows),
        "sales": rng.uniform(100, 1000, size=num_rows).round(2),
        "year": rng.integers(2010, 2024, size=num_rows),
        "product_id": rng.integers(1000, 5000, size=num_rows),
    })
    df_source.to_csv(csv_path, index=False)

    start = time.perf_counter()
    pandas_result = pd.read_csv(csv_path).groupby("country")["sales"].mean()
    pandas_time = time.perf_counter() - start

    print(f"Pandas time: {pandas_time:.4f} s")
    print(pandas_result.sort_index())


    start = time.perf_counter()
    dask_result = dd.read_csv(csv_path).groupby("country")["sales"].mean().compute()
    dask_time = time.perf_counter() - start

    print(f"\nDask time  : {dask_time:.4f} s")
    print(dask_result.sort_index())


Для малого файлу Pandas може бути швидшим, бо Dask має накладні витрати на планування задач. Сенс Dask зʼявляється, коли дані великі або треба паралелити обробку.


## JIT-компіляція: прискорюємо гарячі цикли

Якщо логіку важко векторизувати, можна компілювати функцію під час виконання. Numba добре підходить для числових циклів, де Python-інтерпретатор стає вузьким місцем.


In [ ]:
import time
import numpy as np


def pairwise_distances_python(points: np.ndarray) -> np.ndarray:
    n = len(points)
    result = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dx = points[i][0] - points[j][0]
            dy = points[i][1] - points[j][1]
            result[i, j] = (dx**2 + dy**2) ** 0.5
    return result


N = 2000
points = np.random.default_rng(123).random((N, 2))

start = time.perf_counter()
distances_python = pairwise_distances_python(points)
python_time = time.perf_counter() - start

print(f"Python time: {python_time:.4f} s")
print("Shape:", distances_python.shape)


Цей код спеціально написаний як подвійний цикл. Він зрозумілий, але кожна операція проходить через Python, тому на великих `N` час швидко росте.


In [ ]:
from numba import njit


if njit is not None:
    @njit
    def pairwise_distances_numba(points: np.ndarray) -> np.ndarray:
        n = len(points)
        result = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                dx = points[i][0] - points[j][0]
                dy = points[i][1] - points[j][1]
                result[i, j] = (dx**2 + dy**2) ** 0.5
        return result

    start = time.perf_counter()
    distances_numba_first = pairwise_distances_numba(points)
    first_time = time.perf_counter() - start

    start = time.perf_counter()
    distances_numba_second = pairwise_distances_numba(points)
    second_time = time.perf_counter() - start

    print(f"Numba first run : {first_time:.4f} s")
    print(f"Numba second run: {second_time:.4f} s")
    print("Same result:", np.allclose(distances_python, distances_numba_second))


Перший запуск Numba часто повільніший через компіляцію. Другий запуск показує “гарячий” шлях: функція вже скомпільована для конкретних типів даних.


## JAX: `vmap` і `jit`

JAX будує обчислювальний граф і компілює його через XLA. `vmap` прибирає Python-цикл по батчу, а `jit` компілює обчислення.


In [ ]:
import jax.numpy as jnp
from jax import jit, vmap


def score_one(row):
    return jnp.sum(jnp.sin(row) + row**2 + jnp.log1p(row))

def score_batch_python(data):
    return jnp.array([score_one(row) for row in data])

score_batch_vmap = vmap(score_one)
score_batch_jit = jit(vmap(score_one))

data = jnp.array(np.random.default_rng(0).random((5_000, 64)))

start = time.perf_counter()
result_python = score_batch_python(data).block_until_ready()
python_time = time.perf_counter() - start

start = time.perf_counter()
result_vmap = score_batch_vmap(data).block_until_ready()
vmap_time = time.perf_counter() - start

start = time.perf_counter()
result_jit_first = score_batch_jit(data).block_until_ready()
jit_first_time = time.perf_counter() - start

start = time.perf_counter()
result_jit_second = score_batch_jit(data).block_until_ready()
jit_second_time = time.perf_counter() - start

print(f"Python loop time : {python_time:.4f} s")
print(f"vmap time        : {vmap_time:.4f} s")
print(f"jit first run    : {jit_first_time:.4f} s")
print(f"jit second run   : {jit_second_time:.4f} s")
print("Same result:", bool(jnp.allclose(result_python, result_vmap) and jnp.allclose(result_vmap, result_jit_second)))


У JAX перший `jit`-виклик включає компіляцію, тому він може бути дорогим. Наступні виклики використовують скомпільований граф і зазвичай значно швидші.


## Cython: AOT-компіляція для hot spots

Cython компілює код заздалегідь у C-розширення. Це корисно для стабільних гарячих ділянок, де потрібні C-типи й мінімум Python-overhead.


In [ ]:
CYTHON_PYX = """
def fib_cython(int n):
    cdef int a = 0
    cdef int b = 1
    cdef int i
    for i in range(n):
        a, b = b, a + b
    return a
"""

SETUP_PY = """
from setuptools import setup
from Cython.Build import cythonize

setup(
    ext_modules=cythonize("fibonacci.pyx", language_level=3),
)
"""

RESULT = """
from fibonacci_py import fib_py
import time

N = 10_000_000

start = time.time()
fib_py(N)
print('Python час:', time.time() - start)
"""

print("1. Створюємо файл fibonacci.pyx:")
print(CYTHON_PYX)
print("2. Щоб зібрати це, скомпілюємо за допомогою такого коду (помістити в окремий setup.py):")
print(SETUP_PY)
print("3. Використаймо команду python setup.py build_ext --inplace. Та подивимось результат, запустивши код:")
print(RESULT)


Цю частину краще запускати як окремий мініпроєкт, бо вона створює C-файли й скомпільований модуль. У ноутбуці важливо побачити саму ідею: додаємо C-типи, збираємо extension module, імпортуємо як звичайний Python-модуль. Тож тут лише інструкція, як це запустити, щоб відтворити експеримент.
На такому числі Фібоначчі на Python результат ми б навіть не дочекались, а Cython відпрацює миттєво.


## Zero-copy: `memoryview` і buffer protocol

Zero-copy означає, що ми працюємо з уже існуючим буфером, а не створюємо нову копію байтів. `memoryview` дозволяє “дивитися” на той самий буфер і змінювати його на місці, якщо буфер мутабельний.


In [ ]:
message = bytearray(b"Oops! I broke prod.")
print("Memoryview. До:", message, id(message))

view = memoryview(message)
view[14:18] = b"test"

print("Memoryview. Після:", message, id(message))

message = bytearray(b"Oops! I broke prod.")
print("\nPython replace. До:", message, id(message))

message = message.replace(b"prod", b"test")
print("Python replace. Після:", message, id(message))


У першому випадку змінюється той самий `bytearray`. У другому `replace()` повертає новий обʼєкт, тому `id` змінюється.


In [ ]:
from timeit import timeit
import matplotlib.pyplot as plt


sizes = [10**i for i in range(1, 10)]
byte_times = []
memview_times = []


def slice_bytes(data: bytes):
    _ = data[100:-100]  # операція зрізу


def slice_memview(data: bytes):
    _ = memoryview(data)[100:-100]  # те саме, але zero-copy


for size in sizes:
    data = b'x' * size
    byte_times.append(timeit(lambda: slice_bytes(data), number=10))
    memview_times.append(timeit(lambda: slice_memview(data), number=10))


plt.plot(sizes, byte_times, label='bytes')
plt.plot(sizes, memview_times, label='memoryview')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Size of data (log scale)')
plt.ylabel('Time for slicing (log scale)')
plt.legend()
plt.title('Zero-copy slicing: bytes vs memoryview')
plt.show()


Зріз `bytes` копіює дані, а зріз `memoryview` створює нове представлення на той самий буфер. Для великих байтових блоків це може суттєво зменшити час і памʼять.


In [ ]:
data = bytearray(
    b"1|ANNA |4111111111111111\n"
    b"2|OLEH |5555555555554444\n"
    b"3|MARTA|4000000000000002\n"
)

row_length = len(b"1|ANNA |4111111111111111\n")
card_offset = len(b"1|ANNA |")
card_length = 16

view = memoryview(data)

for row_start in range(0, len(data), row_length):
    card_view = view[row_start + card_offset : row_start + card_offset + card_length]
    card_view[-4:] = b"****"

print(data.decode())


Тут ми маскуємо номери карток без створення окремих рядків для кожного запису. Це приклад роботи з фіксованим байтовим форматом через представлення памʼяті.


## `mmap`: файл як памʼять

`mmap` відображає файл у віртуальну памʼять процесу. Операційна система підвантажує сторінки лише тоді, коли код до них звертається.


In [ ]:
import mmap
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "example.txt"
    path.write_bytes(b"AAAAAAAAAA")

    with path.open("r+b") as file:
        mm = mmap.mmap(file.fileno(), 0)
        print("До:", mm[:])
        mm[0] = ord("B")
        print("Після mmap змін:", mm[:])
        mm.close()

    print("Контент файла на диску:", path.read_bytes())


Зміна через `mmap` відображається у файлі. Це корисно для великих файлів, де не хочеться читати весь вміст у Python-обʼєкт `bytes`.


In [ ]:
import mmap
import tempfile
import time
import tracemalloc
from pathlib import Path


def generate_log_file(filename: Path, line_count: int = 50_000_000) -> None:
    error_positions = {123, 25_000, 99_999, line_count - 1}
    with filename.open("wb") as file:
        for index in range(line_count):
            level = "ERROR" if index in error_positions else "INFO"
            line = f"{index} | {level} | message {index}\n"
            file.write(line.encode())


def find_last_error_read_all(filename: Path) -> str | None:
    data = filename.read_bytes()
    pos = data.rfind(b"ERROR")
    if pos == -1:
        return None

    line_start = data.rfind(b"\n", 0, pos) + 1
    line_end = data.find(b"\n", pos)
    if line_end == -1:
        line_end = len(data)

    return data[line_start:line_end].decode()


def find_last_error_mmap(filename: Path) -> str | None:
    with filename.open("rb") as file:
        with mmap.mmap(file.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            pos = mm.rfind(b"ERROR")
            if pos == -1:
                return None

            line_start = mm.rfind(b"\n", 0, pos) + 1
            line_end = mm.find(b"\n", pos)
            if line_end == -1:
                line_end = len(mm)

            return mm[line_start:line_end].decode()


def measure(func, filename: Path, label: str) -> None:
    tracemalloc.start()
    start = time.perf_counter()

    result = func(filename)

    elapsed = time.perf_counter() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print(f"=== {label} ===")
    print(f"Result: {result}")
    print(f"Time: {elapsed:.6f} s")
    print(f"Peak memory: {peak / 1024 / 1024:.2f} MB")


with tempfile.TemporaryDirectory() as temp_dir:
    log_file = Path(temp_dir) / "app.log"
    generate_log_file(log_file)

    measure(find_last_error_read_all, log_file, "Read whole file")
    print()
    measure(find_last_error_mmap, log_file, "mmap")


`read()` створює повну копію файла в памʼяті Python. `mmap` дозволяє шукати вміст через відображення файла, і пік Python-алокацій зазвичай нижчий.


## Компактні структури даних: dict, Pandas, Arrow

Однакові дані можна зберігати в дуже різних форматах. Список словників гнучкий, але має багато Python-обʼєктів; Pandas і Arrow зберігають дані колонками компактніше.


In [ ]:
from pympler import asizeof
import pandas as pd
import pyarrow as pa

N = 1_000_000

records = [{"id": i, "name": f"name{i}", "age": i % 100} for i in range(N)]
print("List[dict]:", round(asizeof.asizeof(records) / 1024**2, 2), "MB")

df = pd.DataFrame({
    "id": range(N),
    "name": [f"name{i}" for i in range(N)],
    "age": [i % 100 for i in range(N)],
})
print("Pandas DataFrame:", round(df.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

arrow_table = pa.table({
    "id": pa.array(range(N), type=pa.int64()),
    "name": pa.array([f"name{i}" for i in range(N)], type=pa.string()),
    "age": pa.array([i % 100 for i in range(N)], type=pa.int64()),
})
print("PyArrow Table:", round(arrow_table.nbytes / 1024**2, 2), "MB")


Arrow зберігає дані в колонковому бінарному форматі, близькому до того, як їх зручно читати процесору. Для аналітичних задач це часто ефективніше за мільйони окремих Python-словників.


## Підсумок

Не треба відмовлятися від Python. Часто достатньо залишити Python як зручний “клей”, а важкі частини передати NumPy, Dask, Numba, JAX, Cython або zero-copy механізмам. Головне правило: спочатку вимірюємо, потім оптимізуємо.
